In [ ]:
import os
import random
import numpy as np
from glob import glob
from PIL import Image
from tqdm import tqdm
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# ===== Config =====
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

data_dir = '../../data/marker_gene_spatial_expression'
model_path = '../../model/marker_gene_spatial_expression/'
os.makedirs(model_path, exist_ok=True)

NUM_GENES = 15
BATCH_SIZE = 16
NUM_EPOCHS = 1000
LR = 2e-4
NUM_WORKERS = 4
ENCODER_NAME = 'tu-convnext_tiny'

MARKER_GENES = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

In [ ]:
# ===== Augmentation =====
def apply_color_augmentation(image):
    """H&E 병리 이미지용 color augmentation (numpy uint8 HWC)"""
    # Brightness (50%)
    if random.random() < 0.5:
        f = random.uniform(0.8, 1.2)
        image = np.clip(image * f, 0, 255).astype(np.uint8)

    # Contrast (50%)
    if random.random() < 0.5:
        f = random.uniform(0.8, 1.2)
        mean = image.mean()
        image = np.clip((image - mean) * f + mean, 0, 255).astype(np.uint8)

    # Hue shift (50%)
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] = np.clip(hsv[:, :, 0] + random.uniform(-10, 10), 0, 179)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # Saturation (50%)
    if random.random() < 0.5:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.7, 1.3), 0, 255)
        image = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    # Gamma correction (30%)
    if random.random() < 0.3:
        inv_gamma = 1.0 / random.uniform(0.8, 1.2)
        table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype(np.uint8)
        image = cv2.LUT(image, table)

    # Channel-wise brightness (30%)
    if random.random() < 0.3:
        for c in range(3):
            image[:, :, c] = np.clip(
                image[:, :, c].astype(np.float32) + random.uniform(-10, 10), 0, 255
            ).astype(np.uint8)

    # Gaussian noise (30%)
    if random.random() < 0.3:
        noise = np.random.normal(0, random.uniform(0, 5), image.shape)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # Gaussian blur (20%)
    if random.random() < 0.2:
        ks = random.choice([3, 5])
        image = cv2.GaussianBlur(image, (ks, ks), 0)

    return image


# ===== Dataset =====
class MultiScaleRegressionDataset(Dataset):
    def __init__(self, img_20x_list, img_5x_list, label_prop_list, label_int_list, augment=False):
        self.img_20x_paths = img_20x_list
        self.img_5x_paths = img_5x_list
        self.label_prop_paths = label_prop_list
        self.label_int_paths = label_int_list
        self.augment = augment
        self.to_tensor = ToTensor()

    def __len__(self):
        return len(self.img_20x_paths)

    def __getitem__(self, idx):
        img_20x = np.array(Image.open(self.img_20x_paths[idx]).convert('RGB'))
        img_5x = np.array(Image.open(self.img_5x_paths[idx]).convert('RGB'))
        label_prop = torch.from_numpy(np.load(self.label_prop_paths[idx])).float()
        label_int = torch.from_numpy(np.load(self.label_int_paths[idx])).float()

        if self.augment:
            # --- Geometric (동일하게 적용) ---
            # Horizontal flip (50%)
            if random.random() < 0.5:
                img_20x = np.fliplr(img_20x).copy()
                img_5x = np.fliplr(img_5x).copy()

            # Vertical flip (50%)
            if random.random() < 0.5:
                img_20x = np.flipud(img_20x).copy()
                img_5x = np.flipud(img_5x).copy()

            # 90° rotation (30%)
            if random.random() < 0.3:
                k = random.randint(1, 3)
                img_20x = np.rot90(img_20x, k).copy()
                img_5x = np.rot90(img_5x, k).copy()

            # --- Color (각각 독립 적용: 스케일별 staining 차이 반영) ---
            img_20x = apply_color_augmentation(img_20x)
            img_5x = apply_color_augmentation(img_5x)

        img_20x = self.to_tensor(img_20x.copy())
        img_5x = self.to_tensor(img_5x.copy())

        return img_20x, img_5x, label_prop, label_int


# ===== File list =====
img_20x_list = sorted(glob(f'{data_dir}/image/**/*.tiff', recursive=True))
print(f'Total 20x patches: {len(img_20x_list)}')

img_5x_list = [p.replace('/image/', '/image_5x/') for p in img_20x_list]
label_prop_list = [p.replace('/image/', '/label_proportion/').replace('.tiff', '.npy') for p in img_20x_list]
label_int_list = [p.replace('/image/', '/label_intensity/').replace('.tiff', '.npy') for p in img_20x_list]

# 존재하지 않는 파일 필터링
valid_idx = [i for i in range(len(img_20x_list))
             if os.path.exists(img_5x_list[i])
             and os.path.exists(label_prop_list[i])
             and os.path.exists(label_int_list[i])]
print(f'Valid samples: {len(valid_idx)} / {len(img_20x_list)}')

img_20x_list = [img_20x_list[i] for i in valid_idx]
img_5x_list = [img_5x_list[i] for i in valid_idx]
label_prop_list = [label_prop_list[i] for i in valid_idx]
label_int_list = [label_int_list[i] for i in valid_idx]

# Train / Val split
indices = list(range(len(img_20x_list)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

train_dataset = MultiScaleRegressionDataset(
    [img_20x_list[i] for i in train_idx], [img_5x_list[i] for i in train_idx],
    [label_prop_list[i] for i in train_idx], [label_int_list[i] for i in train_idx],
    augment=True
)
val_dataset = MultiScaleRegressionDataset(
    [img_20x_list[i] for i in val_idx], [img_5x_list[i] for i in val_idx],
    [label_prop_list[i] for i in val_idx], [label_int_list[i] for i in val_idx],
    augment=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# ===== Model =====
class MultiScaleRegressionModel(nn.Module):
    def __init__(self, encoder_name='tu-convnext_tiny', num_genes=15):
        super().__init__()
        self.encoder_20x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')
        self.encoder_5x = smp.encoders.get_encoder(encoder_name, in_channels=3, weights='imagenet')

        # ConvNeXt-Tiny last stage output channels
        enc_channels = self.encoder_20x.out_channels  # e.g. (3, 96, 192, 384, 768, 768)
        feat_dim = enc_channels[-1]  # 768

        # Head A: Proportion
        self.head_prop = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
            nn.Sigmoid()
        )

        # Head B: Intensity
        self.head_int = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_genes),
            nn.Sigmoid()
        )

    def forward(self, x_20x, x_5x):
        feat_20x = self.encoder_20x(x_20x)  # list of feature maps
        feat_5x = self.encoder_5x(x_5x)

        # Fuse last stage features
        fused = feat_20x[-1] + feat_5x[-1]  # (B, 768, H, W)

        # Global Average Pooling
        pooled = F.adaptive_avg_pool2d(fused, 1).flatten(1)  # (B, 768)

        prop = self.head_prop(pooled)  # (B, 15)
        intensity = self.head_int(pooled)  # (B, 15)

        return prop, intensity


# ===== Loss =====
def pearson_corr_loss(pred, target):
    """1 - mean PCC across genes. pred, target: (B, G)"""
    pred_c = pred - pred.mean(dim=0, keepdim=True)
    target_c = target - target.mean(dim=0, keepdim=True)
    cov = (pred_c * target_c).sum(dim=0)
    pred_std = pred_c.pow(2).sum(dim=0).sqrt()
    target_std = target_c.pow(2).sum(dim=0).sqrt()
    pcc = cov / (pred_std * target_std + 1e-8)
    return 1.0 - pcc.mean()  # 1 - mean PCC


class TwoHeadLoss(nn.Module):
    """
    Proportion: BCE (0~1 binary-like distribution)
    Intensity:  MSE
    PCC Loss:   1 - Pearson correlation (직접 상관관계 최적화)

    total = α × BCE(prop) + β × MSE(int) + γ × PCC_loss(both heads)
    """
    def __init__(self, prop_weight=1.0, int_weight=1.0, pcc_weight=0.5):
        super().__init__()
        self.prop_weight = prop_weight
        self.int_weight = int_weight
        self.pcc_weight = pcc_weight
        self.bce = nn.BCELoss()
        self.mse = nn.MSELoss()

    def forward(self, pred_prop, pred_int, gt_prop, gt_int):
        loss_prop = self.bce(pred_prop, gt_prop)
        loss_int = self.mse(pred_int, gt_int)

        # PCC loss: 두 head 모두에 적용
        pcc_prop = pearson_corr_loss(pred_prop, gt_prop)
        pcc_int = pearson_corr_loss(pred_int, gt_int)
        loss_pcc = (pcc_prop + pcc_int) / 2.0

        total = (self.prop_weight * loss_prop
                 + self.int_weight * loss_int
                 + self.pcc_weight * loss_pcc)
        return total, loss_prop, loss_int, loss_pcc


# ===== PCC metric =====
def pearson_corr_vector(pred, target):
    """Per-gene Pearson correlation. pred, target: (B, 15) -> returns (15,)"""
    pred_c = pred - pred.mean(dim=0, keepdim=True)
    target_c = target - target.mean(dim=0, keepdim=True)
    cov = (pred_c * target_c).sum(dim=0)
    pred_std = pred_c.pow(2).sum(dim=0).sqrt()
    target_std = target_c.pow(2).sum(dim=0).sqrt()
    pcc = cov / (pred_std * target_std + 1e-8)
    return pcc  # (15,)


model = MultiScaleRegressionModel(ENCODER_NAME, NUM_GENES).to(device)
criterion = TwoHeadLoss(prop_weight=1.0, int_weight=1.0, pcc_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable_params:,}')
print(f'Loss: BCE(prop) + MSE(int) + 0.5 × PCC_loss')

# Shape test
with torch.no_grad():
    dummy_20x = torch.randn(2, 3, 512, 512).to(device)
    dummy_5x = torch.randn(2, 3, 512, 512).to(device)
    p, i = model(dummy_20x, dummy_5x)
    print(f'Output shapes: prop={p.shape}, int={i.shape}')

In [ ]:
# ===== Training Loop =====
train_loss_list, val_loss_list = [], []
train_prop_loss_list, val_prop_loss_list = [], []
train_int_loss_list, val_int_loss_list = [], []
train_pcc_loss_list, val_pcc_loss_list = [], []
train_pcc_prop_list, val_pcc_prop_list = [], []
train_pcc_int_list, val_pcc_int_list = [], []
MIN_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running_loss, running_prop_loss, running_int_loss, running_pcc_loss = 0, 0, 0, 0
    all_pred_prop, all_gt_prop = [], []
    all_pred_int, all_gt_int = [], []

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:4d} [Train]', leave=False)
    for img_20x, img_5x, gt_prop, gt_int in pbar:
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        gt_prop = gt_prop.to(device)
        gt_int = gt_int.to(device)

        optimizer.zero_grad()
        pred_prop, pred_int = model(img_20x, img_5x)
        loss, lp, li, lpcc = criterion(pred_prop, pred_int, gt_prop, gt_int)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_prop_loss += lp.item()
        running_int_loss += li.item()
        running_pcc_loss += lpcc.item()
        all_pred_prop.append(pred_prop.detach().cpu())
        all_gt_prop.append(gt_prop.cpu())
        all_pred_int.append(pred_int.detach().cpu())
        all_gt_int.append(gt_int.cpu())

        pbar.set_postfix(loss=f'{running_loss / (pbar.n + 1):.4f}')

    n_train = len(train_loader)
    train_loss_list.append(running_loss / n_train)
    train_prop_loss_list.append(running_prop_loss / n_train)
    train_int_loss_list.append(running_int_loss / n_train)
    train_pcc_loss_list.append(running_pcc_loss / n_train)

    all_pred_prop = torch.cat(all_pred_prop)
    all_gt_prop = torch.cat(all_gt_prop)
    all_pred_int = torch.cat(all_pred_int)
    all_gt_int = torch.cat(all_gt_int)
    train_pcc_prop_list.append(pearson_corr_vector(all_pred_prop, all_gt_prop).mean().item())
    train_pcc_int_list.append(pearson_corr_vector(all_pred_int, all_gt_int).mean().item())

    # --- Validation ---
    model.eval()
    running_loss, running_prop_loss, running_int_loss, running_pcc_loss = 0, 0, 0, 0
    all_pred_prop, all_gt_prop = [], []
    all_pred_int, all_gt_int = [], []

    with torch.no_grad():
        for img_20x, img_5x, gt_prop, gt_int in tqdm(val_loader, desc=f'Epoch {epoch:4d} [Val]', leave=False):
            img_20x = img_20x.to(device)
            img_5x = img_5x.to(device)
            gt_prop = gt_prop.to(device)
            gt_int = gt_int.to(device)

            pred_prop, pred_int = model(img_20x, img_5x)
            loss, lp, li, lpcc = criterion(pred_prop, pred_int, gt_prop, gt_int)

            running_loss += loss.item()
            running_prop_loss += lp.item()
            running_int_loss += li.item()
            running_pcc_loss += lpcc.item()
            all_pred_prop.append(pred_prop.cpu())
            all_gt_prop.append(gt_prop.cpu())
            all_pred_int.append(pred_int.cpu())
            all_gt_int.append(gt_int.cpu())

    n_val = len(val_loader)
    val_total = running_loss / n_val
    val_loss_list.append(val_total)
    val_prop_loss_list.append(running_prop_loss / n_val)
    val_int_loss_list.append(running_int_loss / n_val)
    val_pcc_loss_list.append(running_pcc_loss / n_val)

    all_pred_prop = torch.cat(all_pred_prop)
    all_gt_prop = torch.cat(all_gt_prop)
    all_pred_int = torch.cat(all_pred_int)
    all_gt_int = torch.cat(all_gt_int)
    val_pcc_prop = pearson_corr_vector(all_pred_prop, all_gt_prop).mean().item()
    val_pcc_int = pearson_corr_vector(all_pred_int, all_gt_int).mean().item()
    val_pcc_prop_list.append(val_pcc_prop)
    val_pcc_int_list.append(val_pcc_int)
    print(f'Epoch {epoch:4d} | '
            f'Train: {train_loss_list[-1]:.4f} (BCE:{train_prop_loss_list[-1]:.4f} MSE:{train_int_loss_list[-1]:.4f} PCC:{train_pcc_loss_list[-1]:.4f}) | '
            f'Val: {val_loss_list[-1]:.4f} (BCE:{val_prop_loss_list[-1]:.4f} MSE:{val_int_loss_list[-1]:.4f} PCC:{val_pcc_loss_list[-1]:.4f}) | '
            f'PCC prop: {val_pcc_prop:.4f} int: {val_pcc_int:.4f}')    
    # --- Checkpoint ---
    if val_total < MIN_loss:
        MIN_loss = val_total
        torch.save(model.state_dict(), f'{model_path}train_2head_best.pt')




    # --- Plot ---
    if epoch % 100 == 99:
        fig, axes = plt.subplots(1, 4, figsize=(24, 5))

        ax = axes[0]
        ax.plot(train_loss_list, label='Train Total')
        ax.plot(val_loss_list, label='Val Total')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Total Loss'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[1]
        ax.plot(train_prop_loss_list, label='Train BCE(prop)', alpha=0.7)
        ax.plot(val_prop_loss_list, label='Val BCE(prop)', alpha=0.7)
        ax.plot(train_int_loss_list, label='Train MSE(int)', alpha=0.7, linestyle='--')
        ax.plot(val_int_loss_list, label='Val MSE(int)', alpha=0.7, linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Per-Head Loss'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[2]
        ax.plot(train_pcc_loss_list, label='Train PCC Loss', alpha=0.7)
        ax.plot(val_pcc_loss_list, label='Val PCC Loss', alpha=0.7)
        ax.set_xlabel('Epoch'); ax.set_ylabel('1 - PCC')
        ax.set_title('PCC Loss (1 - corr)'); ax.legend(); ax.grid(True, alpha=0.3)

        ax = axes[3]
        ax.plot(train_pcc_prop_list, label='Train Prop PCC', alpha=0.7)
        ax.plot(val_pcc_prop_list, label='Val Prop PCC', alpha=0.7)
        ax.plot(train_pcc_int_list, label='Train Int PCC', alpha=0.7, linestyle='--')
        ax.plot(val_pcc_int_list, label='Val Int PCC', alpha=0.7, linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('PCC')
        ax.set_title('Pearson Correlation'); ax.legend(); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

print(f'\nBest val loss: {MIN_loss:.6f}')
print(f'Model saved: {model_path}train_2head_best.pt')

In [ ]:
# ===== Evaluation =====
model.load_state_dict(torch.load(f'{model_path}train_2head_best.pt', map_location=device))
model.eval()

all_pred_prop, all_gt_prop = [], []
all_pred_int, all_gt_int = [], []

with torch.no_grad():
    for img_20x, img_5x, gt_prop, gt_int in val_loader:
        img_20x = img_20x.to(device)
        img_5x = img_5x.to(device)
        pred_prop, pred_int = model(img_20x, img_5x)
        all_pred_prop.append(pred_prop.cpu())
        all_gt_prop.append(gt_prop)
        all_pred_int.append(pred_int.cpu())
        all_gt_int.append(gt_int)

all_pred_prop = torch.cat(all_pred_prop)
all_gt_prop = torch.cat(all_gt_prop)
all_pred_int = torch.cat(all_pred_int)
all_gt_int = torch.cat(all_gt_int)

pcc_prop = pearson_corr_vector(all_pred_prop, all_gt_prop)  # (15,)
pcc_int = pearson_corr_vector(all_pred_int, all_gt_int)  # (15,)
mae_prop = (all_pred_prop - all_gt_prop).abs().mean(dim=0)  # (15,)
mae_int = (all_pred_int - all_gt_int).abs().mean(dim=0)  # (15,)

print(f'{"Gene":>10s} | {"PCC(prop)":>10s} {"MAE(prop)":>10s} | {"PCC(int)":>10s} {"MAE(int)":>10s}')
print('-' * 60)
for gi, gene in enumerate(MARKER_GENES):
    print(f'{gene:>10s} | {pcc_prop[gi]:10.4f} {mae_prop[gi]:10.4f} | {pcc_int[gi]:10.4f} {mae_int[gi]:10.4f}')
print('-' * 60)
print(f'{"Mean":>10s} | {pcc_prop.mean():10.4f} {mae_prop.mean():10.4f} | {pcc_int.mean():10.4f} {mae_int.mean():10.4f}')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(NUM_GENES)
w = 0.35

ax = axes[0]
ax.bar(x - w/2, pcc_prop.numpy(), w, label='Proportion', color='steelblue')
ax.bar(x + w/2, pcc_int.numpy(), w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('PCC'); ax.set_title('Per-Gene Pearson Correlation')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linewidth=0.5)

ax = axes[1]
ax.bar(x - w/2, mae_prop.numpy(), w, label='Proportion', color='steelblue')
ax.bar(x + w/2, mae_int.numpy(), w, label='Intensity', color='coral')
ax.set_xticks(x); ax.set_xticklabels(MARKER_GENES, rotation=45, ha='right')
ax.set_ylabel('MAE'); ax.set_title('Per-Gene Mean Absolute Error')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()